# Reusable PyTorch Training Pipeline

Goal: combine the training concepts learned so far into reusable functions.

Features:
- training and validation loops
- GPU support
- early stopping
- learning-rate scheduling
- loss history
- best-model restoration

In [1]:
import copy
import torch
from torch import nn

In [2]:
# reusable training function

def train_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device,
    epochs=500,
    scheduler=None,
    patience=None
):
    train_losses = []
    val_losses = []
    learning_rates = []

    best_val_loss = float("inf")
    best_model_state = None
    epochs_without_improvement = 0

    for epoch in range(epochs):

        # -------------------
        # Training
        # -------------------

        model.train()
        train_loss = 0.0

        for xb, yb in train_loader:

            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()

            predictions = model(xb)

            loss = criterion(predictions, yb)

            loss.backward()

            optimizer.step()

            train_loss += loss.item()

        train_loss /= len(train_loader)

        # -------------------
        # Validation
        # -------------------

        model.eval()
        val_loss = 0.0

        with torch.no_grad():

            for xb, yb in val_loader:

                xb = xb.to(device)
                yb = yb.to(device)

                predictions = model(xb)

                loss = criterion(predictions, yb)

                val_loss += loss.item()

        val_loss /= len(val_loader)

        # -------------------
        # Record metrics
        # -------------------

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        current_lr = optimizer.param_groups[0]["lr"]
        learning_rates.append(current_lr)

        # -------------------
        # Save best model
        # -------------------

        if val_loss < best_val_loss:

            best_val_loss = val_loss
            best_model_state = copy.deepcopy(
                model.state_dict()
            )

            epochs_without_improvement = 0

        else:

            epochs_without_improvement += 1

        # -------------------
        # Learning-rate scheduler
        # -------------------

        if scheduler is not None:
            scheduler.step(val_loss)

        # -------------------
        # Progress
        # -------------------

        if epoch % 50 == 0:

            print(
                f"Epoch {epoch:4d} | "
                f"Train: {train_loss:.4f} | "
                f"Val: {val_loss:.4f} | "
                f"LR: {current_lr:.6f}"
            )

        # -------------------
        # Early stopping
        # -------------------

        if (
            patience is not None
            and epochs_without_improvement >= patience
        ):

            print(
                f"\nEarly stopping at epoch {epoch}"
            )

            break

    # Restore best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    history = {
        "train_loss": train_losses,
        "val_loss": val_losses,
        "learning_rate": learning_rates,
        "best_val_loss": best_val_loss
    }

    return model, history

In [3]:
model = BatchNormNetwork().to(device)

criterion = nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

NameError: name 'BatchNormNetwork' is not defined